# Limb Sounding Observations

Limb-sounding instruments observe the atmosphere by looking sideways, toward the Earth's limb, rather than straight down at the surface. Like [GNSS Radio Occultation](CollectRO.ipynb), the observation geometry is characterized by a *tangent point*: the point along the viewing ray with minimum distance to the geocenter. Unlike RO, which relies on the geometry between two independently-orbiting satellites (a receiver and a transmitter), a limb sounder's viewing ray is defined entirely by its own host spacecraft: a fixed mounting direction relative to the velocity vector, actively scanned in elevation to sweep the tangent point up and down through the atmosphere and build a vertical profile.

This TAT-C analysis function, `collect_limb_observations`, models where and when those tangent points fall as a limb-sounding spacecraft flies its orbit. As with `collect_ro_observations`, this analysis does not model atmospheric refraction and represents only a coarse geometric estimate suitable for early-stage mission analysis.

## Mathematical Basis

### Viewing Geometry

Let the satellite have position $x_{sat}$ and velocity $v_{sat}$ in a geocentric inertial frame. Define the body-fixed VNB frame (shared with the RO analysis):

$\hat{v} = \frac{v_{sat}}{||v_{sat}||}, \qquad \hat{n} = \frac{x_{sat} \times v_{sat}}{||x_{sat} \times v_{sat}||}, \qquad \hat{b} = \hat{v} \times \hat{n}$

$\hat{v}$ points along the velocity, $\hat{n}$ is normal to the orbit plane (and therefore always exactly horizontal, perpendicular to $x_{sat}$), and $\hat{b}$ completes the right-handed frame, coinciding with the zenith direction $x_{sat}/||x_{sat}||$ for a circular orbit.

The sensor's viewing direction is set by an azimuth $\psi$ (fixed, relative to velocity, measured in the local horizontal plane) and an elevation $\epsilon$ (actively scanned, the depression angle below local horizontal):

$\hat{d} = \cos\epsilon \left( \cos\psi\, \hat{v} + \sin\psi\, \hat{n} \right) - \sin\epsilon\, \hat{b}$

The tangent point is the closest approach of the ray $x_{sat} + s\hat{d}$ to the origin, i.e. the foot of the perpendicular from the geocenter:

$x_{tp} = x_{sat} - \left( x_{sat} \cdot \hat{d} \right) \hat{d}$

Note that this is a *geocentric* tangent point: the ray's point of minimum distance to the Earth's center (the origin), not the point where the ray is exactly tangent to the WGS 84 reference ellipsoid's surface. The two definitions coincide at the equator and poles but diverge slightly elsewhere, since the ellipsoid's surface normal is not generally parallel to the geocentric radius vector; the elevation reported below is simply the WGS 84 conversion of this geocentric point.

### Choosing the Scan Angle

For a spherical Earth, the tangent point's geocentric radius depends only on the angle between $\hat{d}$ and $x_{sat}$, independent of azimuth: $r_{tp} = ||x_{sat}|| \cos\epsilon$. Inverting this gives the (approximate) elevation angle needed to reach a target tangent point elevation $h$ above a mean Earth radius $R_E$:

$\epsilon \approx \arccos\left( \frac{R_E + h}{||x_{sat}||} \right)$

This approximation is used only to *choose* the scan angle at each sample; the tangent point actually reported is always computed exactly (WGS 84, via Skyfield), so achieved elevations can differ slightly from the requested targets -- the same spirit as the `sample_elevation` interpolation used by `collect_ro_observations`.

## Analysis

First, we configure the vertical scan. Four parameters set the sensor's viewing geometry:
 * `scan_azimuth` sets the sensor's fixed mounting direction (in degrees) relative to the velocity vector, measured in the local horizontal plane (0 = forward, 90 = cross-track).
 * `scan_elevations` sets the target tangent point elevations (in meters) that make up one vertical scan. Order doesn't matter here -- `scan_direction` (below) determines the sweep order -- and samples are timed across `scan_duration` as a constant angular-rate scan mirror would reach them, so unevenly-spaced elevations are not visited at evenly-spaced times.
 * `scan_duration` sets the total time to sweep through `scan_elevations`, starting at each requested scan time -- since the spacecraft keeps moving throughout the scan, this determines how much the tangent point drifts along track during a single profile.
 * `scan_direction` sets which of the two broad classes of real limb sounder scan is modeled: `UPWARD` (lowest elevation first, e.g. MLS) or `DOWNWARD` (highest first, e.g. SABER). Left unset, it defaults from `scan_azimuth`: forward-looking sensors (azimuth closer to 0 deg than 180 deg) default to `UPWARD`, rearward-looking ones default to `DOWNWARD`.

These parameters approximate the geometry of [MLS](https://mls.jpl.nasa.gov/) (Microwave Limb Sounder), an instrument aboard the Aura spacecraft that views forward, along the flight direction, and scans bottom-to-top (upward) from the surface to roughly 90 km altitude in an approximately 25-second cycle (about 3,500 scans per day). Since MLS looks forward, `scan_direction` doesn't need to be set explicitly below -- it defaults to `UPWARD` automatically.

Science applications typically specify vertical levels by atmospheric pressure rather than altitude. `tatc.utils.pressure_to_altitude` converts between the two using the U.S. Standard Atmosphere (1976) model, so `scan_elevations` can be specified directly from a standard pressure grid, similar to how MLS data products are reported.

In [ ]:
from datetime import datetime, timedelta, timezone
import numpy as np
from tatc.utils import pressure_to_altitude

# limb scan configuration
scan_azimuth = 0  # deg, forward (along-track) view -- defaults scan_direction to UPWARD
scan_pressures_hpa = [1000, 300, 100, 30, 10, 3, 1, 0.3, 0.1, 0.03, 0.01]
scan_elevations = [pressure_to_altitude(p * 100) for p in scan_pressures_hpa]  # m
scan_duration = timedelta(seconds=25)  # time to complete one vertical scan

# scenario configuration
start = datetime(2026, 8, 14, tzinfo=timezone.utc)
duration = timedelta(days=1)
end = start + duration
times = [start + i * scan_duration for i in range(duration // scan_duration)]
print(f"{len(times)} scans requested over {duration}")

Next, we define the spacecraft carrying the limb sounder. Aura has flown a sun-synchronous orbit at approximately 700 km altitude and 98.3 degree inclination since 2004, as part of the A-Train constellation. The code below builds a TAT-C `Satellite` from a recent two-line element (TLE) set; note this example follows `collect_ro_observations` in taking plain numeric arguments to describe the sensor's viewing geometry, rather than an `Instrument` schema.

In [ ]:
from tatc.schemas import GeneralPerturbationsOrbit, Satellite

aura = Satellite(
    name="Aura",
    orbit=GeneralPerturbationsOrbit.from_tle(
        [
            "1 28376U 04026A   26226.59599742  .00000334  00000+0  77326-4 0  9994",
            "2 28376  98.3460 183.5488 0001688  86.3454 273.7940 14.61403667174741",
        ]
    ),
)

TAT-C simulates limb sounding observations using the `collect_limb_observations` function. For each requested start time, it samples one satellite position per `scan_elevations` entry, timed across `scan_duration` per `scan_direction` as described above, and computes that sample's tangent point.

The outputs report each vertical scan including the following fields:
 * `satellite`: the satellite name
 * `geometry`: a multipoint geometry describing the full swept tangent point track for one scan, in sweep order
 * `position`: a point geometry describing the scan interpolated at the `sample_elevation` value (defaults to the midpoint of `scan_elevations`)
 * `start`: time of the first valid tangent point in the scan
 * `end`: time of the last valid tangent point in the scan
 * `time`: time of the point interpolated at `sample_elevation`

In [ ]:
from tatc.analysis import collect_limb_observations

limb_obs = collect_limb_observations(
    aura, times, scan_azimuth, scan_elevations, scan_duration
)
display(limb_obs)

The limb observations can be better visualized by combining with the satellite's orbit track.

In [ ]:
from tatc.analysis import collect_orbit_track

orbit_track = collect_orbit_track(aura, times)

The resulting plot shows the Aura orbit track together with every sampled tangent point, colored by elevation.

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

lons = np.concatenate([[p.x for p in g.geoms] for g in limb_obs.geometry])
lats = np.concatenate([[p.y for p in g.geoms] for g in limb_obs.geometry])
elevs = np.concatenate([[p.z for p in g.geoms] for g in limb_obs.geometry])

fig, ax = plt.subplots(figsize=(10, 4), subplot_kw={"projection": ccrs.PlateCarree()})
orbit_track.plot(
    ax=ax, marker=".", color="k", markersize=1, lw=0, transform=ccrs.PlateCarree()
)
sc = ax.scatter(
    lons, lats, c=elevs / 1e3, cmap="viridis", s=2, transform=ccrs.PlateCarree()
)
fig.colorbar(sc, ax=ax, orientation="vertical", pad=0.02, label="Tangent Point Elevation (km)")
ax.stock_img()
ax.set_global()
ax.set_title("Aura Orbit Track and Limb Tangent Points")
plt.show()

Each row's `geometry` traces a single vertical scan. Plotting one scan's tangent point elevation against its sample index shows the vertical profile through the atmosphere that a limb sounder builds as it sweeps `scan_elevations`.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))

sample_scan = limb_obs.iloc[len(limb_obs) // 2].geometry
heights_km = [p.z / 1e3 for p in sample_scan.geoms]

ax.plot(heights_km, range(len(heights_km)), marker="o")
ax.set_xlabel("Tangent Point Elevation (km)")
ax.set_ylabel("Scan Sample Index")
ax.set_title("Single Vertical Scan Profile")
plt.show()

An animation can show the dynamic collection of vertical scans aggregated to individual (hourly) frames.

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(8, 4), subplot_kw={"projection": ccrs.PlateCarree()})

frame_duration = timedelta(hours=1)
num_frames = int(duration / frame_duration)


def animate(frame):
    ax.clear()
    time = times[0] + frame * frame_duration
    orbit_track.loc[
        (orbit_track.time >= time) & (orbit_track.time < time + frame_duration)
    ].plot(ax=ax, marker=".", color="k", markersize=1, lw=0, transform=ccrs.PlateCarree())
    frame_obs = limb_obs.loc[
        (limb_obs.start >= time) & (limb_obs.start < time + frame_duration)
    ]
    if not frame_obs.empty:
        frame_lons = np.concatenate([[p.x for p in g.geoms] for g in frame_obs.geometry])
        frame_lats = np.concatenate([[p.y for p in g.geoms] for g in frame_obs.geometry])
        frame_elevs = np.concatenate([[p.z for p in g.geoms] for g in frame_obs.geometry])
        ax.scatter(
            frame_lons,
            frame_lats,
            c=frame_elevs / 1e3,
            cmap="viridis",
            vmin=0,
            vmax=90,
            s=4,
            transform=ccrs.PlateCarree(),
        )
    ax.set_global()
    ax.coastlines()
    ax.set_aspect("equal")
    ax.set_title(
        time.strftime("%x %X") + "$-$" + (time + frame_duration).strftime("%X")
    )
    fig.tight_layout()


ani = animation.FuncAnimation(fig, animate, frames=num_frames, interval=200, blit=False)
display(HTML(ani.to_jshtml()))
plt.close()